In [4]:
# import

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
# Loading cleaned csv data

df = pd.read_csv('E:/Git/blore-house-data-with-gradient-boost-and-XGBoost/clean_data.csv')
df.isnull().sum()

area             0
location         0
bhk              0
bath             0
balcony          0
parking          0
furnishing       0
property_type    0
age              0
price            0
dtype: int64

### Model Creation

In [ ]:
# defining x and y

x = df.drop(columns='price')
y = np.log1p(df['price'])
x.shape

(981, 9)

In [ ]:
# Train Test split

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.30, random_state=42)

In [30]:
# model

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score


gbr = GradientBoostingRegressor()

scores1 = cross_val_score(
    gbr,
    x_train,
    y_train,
    cv=5,
    scoring='r2'
)

print("CV Scores:", scores1)
print("Mean CV Score:", scores1.mean())

gbr.fit(x_train, y_train)


y_pred1 = gbr.predict(x_test)



CV Scores: [0.61150747 0.50579345 0.5330337  0.63699966 0.57003659]
Mean CV Score: 0.5714741746519068


In [31]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MAE:", mean_absolute_error(y_test, y_pred1))
print("MSE:", mean_squared_error(y_test, y_pred1))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred1)))
print("R2 Score:", r2_score(y_test, y_pred1))


MAE: 0.239581601404157
MSE: 0.08281346999499603
RMSE: 0.28777329618120584
R2 Score: 0.5394632597920254


In [37]:
# XGBoost Model
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score

xgb = XGBRegressor(n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42)

scores2 = cross_val_score(
    xgb,
    x_train,
    y_train,
    cv=5,
    scoring='r2'
)

print("CV Scores:", scores2)
print("Mean CV Score:", scores2.mean())

xgb.fit(x_train, y_train)
y_pred2 = xgb.predict(x_test)

CV Scores: [0.59341851 0.45441549 0.49579725 0.55859769 0.56271061]
Mean CV Score: 0.5329879100376693


In [38]:
print("MAE:", mean_absolute_error(y_test, y_pred2))
print("MSE:", mean_squared_error(y_test, y_pred2))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred2)))
print("R2 Score:", r2_score(y_test, y_pred2))

MAE: 0.24000911221954385
MSE: 0.08746687426830879
RMSE: 0.29574799114839107
R2 Score: 0.5135850586366961


#### Hyperparameter Tuning

In [40]:
from sklearn.model_selection import GridSearchCV

params = {
    'n_estimators': [100, 200, 300, 400, 500, 600, 700],
    'learning_rate': [0.001, 0.01, 0.05, 0.1],
    'max_depth': [1, 2, 3, 4, 5, 6],
    'subsample': [0.7, 0.8, 1.0]
}

grid = GridSearchCV(
    estimator=GradientBoostingRegressor(),
    param_grid=params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(x_train, y_train)

best_model1 = grid.best_estimator_

In [41]:
y_pred3 = best_model1.predict(x_test)

In [42]:
print("MAE:", mean_absolute_error(y_test, y_pred3))
print("MSE:", mean_squared_error(y_test, y_pred3))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred3)))
print("R2 Score:", r2_score(y_test, y_pred3))

MAE: 0.2331718038523454
MSE: 0.07706909489015888
RMSE: 0.27761321094313735
R2 Score: 0.5714084950958158


In [43]:
from sklearn.model_selection import GridSearchCV

params = {
    'n_estimators': [100, 200, 300, 400, 500, 600],
    'learning_rate': [0.001, 0.01, 0.05, 0.1],
    'max_depth': [1, 2, 3, 4, 5],
    'subsample': [0.7, 0.8, 1.0]
}

grid = GridSearchCV(
    estimator=XGBRegressor(),
    param_grid=params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(x_train, y_train)

best_model2 = grid.best_estimator_

In [44]:
y_pred4 = best_model2.predict(x_test)

In [45]:
print("MAE:", mean_absolute_error(y_test, y_pred4))
print("MSE:", mean_squared_error(y_test, y_pred4))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred4)))
print("R2 Score:", r2_score(y_test, y_pred4))

MAE: 0.2348037756363883
MSE: 0.07943914965909936
RMSE: 0.2818495159816659
R2 Score: 0.5582283047539769


In [51]:
# stacking hyper tuned model
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression

# base models
base_models = [
    ('gbr', best_model1),
    ('xgb', best_model2)
]

# meta model
meta_model = LinearRegression()

# stacking model
stacked_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)

# Train
stacked_model.fit(x_train, y_train)

# Predict
y_pred_stack = stacked_model.predict(x_test)

In [52]:
print("MAE:", mean_absolute_error(y_test, y_pred_stack))
print("MSE:", mean_squared_error(y_test, y_pred_stack))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_stack)))
print("R2 Score:", r2_score(y_test, y_pred_stack))

MAE: 0.23413507924500013
MSE: 0.07840866695137233
RMSE: 0.28001547627117385
R2 Score: 0.5639589563869295


In [61]:
# saving model using joblib
import joblib

joblib.dump(stacked_model, 'E:/Git/blore-house-data-with-gradient-boost-and-XGBoost/xgb_gbr_stacked_model.pkl')

['E:/Git/blore-house-data-with-gradient-boost-and-XGBoost/xgb_gbr_stacked_model.pkl']